# Multi-Model Comparison with MLflow

## Topic Goal

This notebook properly implements the expected **Multi-Model Comparison** use case with MLflow.

In real machine learning projects, we rarely train only one model and stop.  
We usually compare multiple algorithms, track their results, select the best one, and then register only the best-performing model.

This notebook demonstrates:

1. Load customer churn dataset.
2. Prepare common preprocessing logic.
3. Define multiple candidate models.
4. Train each model using the same train/test split.
5. Log each candidate model as a separate MLflow comparison run.
6. Save a comparison table.
7. Select the best model based on F1-score.
8. Train/log the best model with input example and signature.
9. Create `model_uri` from the same final best-model run.
10. Register only the best model using that same `model_uri`.

This is a production-style pattern for model benchmarking and model selection.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn models, and visualization libraries.

The notebook compares multiple algorithms using the same preprocessing pipeline.


In [ ]:
# Import os to create folders and manage local paths.
import os

# Import json to save comparison metadata.
import json

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import datetime to timestamp comparison reports.
from datetime import datetime

# Import MLflow for experiment tracking, model logging, and model registry.
import mlflow

# Import MLflow scikit-learn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input/output schema.
from mlflow.models.signature import infer_signature

# Import pandas for tabular data processing.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import matplotlib for comparison charts.
import matplotlib.pyplot as plt

# Import train_test_split for creating train/test split.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing numerical and categorical columns.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import candidate algorithms for comparison.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Local Folders

This notebook uses local MLflow tracking.

Each candidate model gets its own MLflow run.  
The final best model also gets a separate final run with signature, input example, and registry.


In [ ]:
# Define experiment name.
EXPERIMENT_NAME = "advanced_16_multi_model_comparison"

# Define final best-model run name.
RUN_NAME = "16_multi_model_comparison_best_model_same_run"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for comparison reports.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure local MLflow tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Final run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


## 3. Load Dataset

We use the customer churn dataset.

The target column is:

```text
churn
```


In [ ]:
# Check whether dataset exists.
if not os.path.exists(DATA_PATH):
    # Raise clear error if dataset is missing.
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Load dataset.
df = pd.read_csv(DATA_PATH)

# Display first five rows.
display(df.head())

# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Print dataset shape.
print("Dataset shape:", df.shape)


## 4. Prepare Features, Target, and Preprocessing

All models must be compared fairly.

Therefore, every model will use:

- the same dataset
- the same train/test split
- the same preprocessing pipeline
- the same evaluation metrics


In [ ]:
# Create feature matrix by removing ID and target columns.
X = df.drop(columns=[id_column, target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split dataset into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature details.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 5. Define Candidate Models

This is the core of the multi-model comparison use case.

We define multiple algorithms and compare them under the same conditions.


In [ ]:
# Define candidate models for comparison.
candidate_models = {
    # Linear baseline model.
    "logistic_regression": LogisticRegression(max_iter=1000),

    # Tree ensemble model.
    "random_forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        random_state=42
    ),

    # Boosting model.
    "gradient_boosting": GradientBoostingClassifier(
        random_state=42
    ),

    # Extra randomized trees model.
    "extra_trees": ExtraTreesClassifier(
        n_estimators=150,
        max_depth=8,
        random_state=42
    ),

    # Distance-based model.
    "knn": KNeighborsClassifier(
        n_neighbors=7
    ),
}

# Print model names.
print("Candidate models:")
for model_name in candidate_models:
    print("-", model_name)


## 6. Train and Log Each Candidate Model

Each candidate model is trained and logged as a separate MLflow run.

This helps students see multiple runs in MLflow UI and compare metrics easily.


In [ ]:
# Create empty list to collect model comparison results.
comparison_results = []

# Loop through each candidate model.
for model_name, estimator in candidate_models.items():

    # Create pipeline for this model.
    candidate_pipeline = Pipeline(steps=[
        # Apply the common preprocessing.
        ("preprocessor", preprocessor),

        # Train the candidate model.
        ("model", estimator),
    ])

    # Train candidate pipeline.
    candidate_pipeline.fit(X_train, y_train)

    # Predict test data.
    candidate_predictions = candidate_pipeline.predict(X_test)

    # Calculate accuracy.
    candidate_accuracy = accuracy_score(y_test, candidate_predictions)

    # Calculate precision.
    candidate_precision = precision_score(y_test, candidate_predictions, zero_division=0)

    # Calculate recall.
    candidate_recall = recall_score(y_test, candidate_predictions, zero_division=0)

    # Calculate F1-score.
    candidate_f1 = f1_score(y_test, candidate_predictions, zero_division=0)

    # Create classification report.
    candidate_report = classification_report(y_test, candidate_predictions, zero_division=0)

    # Save classification report for this model.
    report_path = f"outputs/{model_name}_classification_report.txt"
    with open(report_path, "w") as file:
        file.write(candidate_report)

    # Start an MLflow run for this candidate model.
    with mlflow.start_run(run_name=f"candidate_{model_name}"):

        # Log run type.
        mlflow.log_param("run_type", "candidate_model_comparison")

        # Log model name.
        mlflow.log_param("model_name", model_name)

        # Log model class.
        mlflow.log_param("model_class", estimator.__class__.__name__)

        # Log dataset path.
        mlflow.log_param("dataset", DATA_PATH)

        # Log accuracy.
        mlflow.log_metric("accuracy", candidate_accuracy)

        # Log precision.
        mlflow.log_metric("precision", candidate_precision)

        # Log recall.
        mlflow.log_metric("recall", candidate_recall)

        # Log F1-score.
        mlflow.log_metric("f1_score", candidate_f1)

        # Log classification report artifact.
        mlflow.log_artifact(report_path)

    # Store comparison result.
    comparison_results.append({
        "model_name": model_name,
        "model_class": estimator.__class__.__name__,
        "accuracy": candidate_accuracy,
        "precision": candidate_precision,
        "recall": candidate_recall,
        "f1_score": candidate_f1,
    })

# Convert comparison results to dataframe.
comparison_df = pd.DataFrame(comparison_results)

# Sort by F1-score descending.
comparison_df = comparison_df.sort_values("f1_score", ascending=False)

# Display model comparison table.
display(comparison_df)


## 7. Save Model Comparison Artifacts

We save:

- comparison CSV
- comparison JSON
- model ranking chart

These artifacts help explain why one model was selected.


In [ ]:
# Save comparison table as CSV.
comparison_df.to_csv("outputs/model_comparison_results.csv", index=False)

# Save comparison table as JSON.
comparison_df.to_json("outputs/model_comparison_results.json", orient="records", indent=4)

# Create model comparison chart.
plt.figure(figsize=(10, 6))
plt.bar(comparison_df["model_name"], comparison_df["f1_score"])
plt.title("Model Comparison by F1-score")
plt.xlabel("Model")
plt.ylabel("F1-score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

# Save chart.
plt.savefig("outputs/model_comparison_f1_chart.png")
plt.close()

# Create comparison metadata.
comparison_metadata = {
    "created_at": datetime.now().isoformat(),
    "selection_metric": "f1_score",
    "candidate_models": list(candidate_models.keys()),
    "best_model": comparison_df.iloc[0]["model_name"],
    "best_f1_score": float(comparison_df.iloc[0]["f1_score"]),
}

# Save comparison metadata.
with open("outputs/model_comparison_metadata.json", "w") as file:
    json.dump(comparison_metadata, file, indent=4)

# Print saved artifacts.
print("Saved comparison artifacts:")
print("- outputs/model_comparison_results.csv")
print("- outputs/model_comparison_results.json")
print("- outputs/model_comparison_f1_chart.png")
print("- outputs/model_comparison_metadata.json")


## 8. Select Best Model

The best model is selected based on F1-score.

In many business cases, F1-score is useful when we care about balance between precision and recall.


In [ ]:
# Select best row from comparison dataframe.
best_row = comparison_df.iloc[0]

# Extract best model name.
best_model_name = best_row["model_name"]

# Extract best model estimator from candidate model dictionary.
best_estimator = candidate_models[best_model_name]

# Print best model details.
print("Best model name:", best_model_name)
print("Best model class:", best_estimator.__class__.__name__)
print("Best F1-score:", best_row["f1_score"])


## 9. Train Final Best Model

Now we retrain the best model as the final selected model.

This final model will be logged with:

- model signature
- input example
- model URI
- registry entry


In [ ]:
# Create final best model pipeline.
best_pipeline = Pipeline(steps=[
    # Apply common preprocessing.
    ("preprocessor", preprocessor),

    # Train selected best estimator.
    ("model", best_estimator),
])

# Train final best pipeline.
best_pipeline.fit(X_train, y_train)

# Generate final predictions.
best_predictions = best_pipeline.predict(X_test)

# Calculate final accuracy.
best_accuracy = accuracy_score(y_test, best_predictions)

# Calculate final precision.
best_precision = precision_score(y_test, best_predictions, zero_division=0)

# Calculate final recall.
best_recall = recall_score(y_test, best_predictions, zero_division=0)

# Calculate final F1-score.
best_f1 = f1_score(y_test, best_predictions, zero_division=0)

# Create final classification report.
best_report = classification_report(y_test, best_predictions, zero_division=0)

# Save final classification report.
with open("outputs/best_model_classification_report.txt", "w") as file:
    file.write(best_report)

# Print final metrics.
print("Best model accuracy:", best_accuracy)
print("Best model precision:", best_precision)
print("Best model recall:", best_recall)
print("Best model F1-score:", best_f1)
print(best_report)


## 10. Infer Signature for Best Model

The signature is inferred only for the final selected best model.

This is the model that will be registered.


In [ ]:
# Create input example.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = best_pipeline.predict(input_example)

# Infer model signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 11. Same-Run MLflow Logging for Final Best Model

This final run logs:

- selected best model name
- comparison artifacts
- final metrics
- final model with signature
- input example
- model URI

Then it registers only the selected best model.


In [ ]:
# Start the SAME MLflow run for the final best model.
with mlflow.start_run(run_name=RUN_NAME):

    # Log run type.
    mlflow.log_param("run_type", "final_best_model")

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log selection metric.
    mlflow.log_param("selection_metric", "f1_score")

    # Log selected best model name.
    mlflow.log_param("best_model_name", best_model_name)

    # Log selected best model class.
    mlflow.log_param("best_model_class", best_estimator.__class__.__name__)

    # Log number of candidate models.
    mlflow.log_param("candidate_model_count", len(candidate_models))

    # Log candidate model names.
    mlflow.log_param("candidate_models", ",".join(candidate_models.keys()))

    # Set searchable tag for run purpose.
    mlflow.set_tag("run_purpose", "multi_model_comparison_best_model_selection")

    # Set searchable tag for best model.
    mlflow.set_tag("best_model_name", best_model_name)

    # Log final best model metrics.
    mlflow.log_metric("accuracy", best_accuracy)
    mlflow.log_metric("precision", best_precision)
    mlflow.log_metric("recall", best_recall)
    mlflow.log_metric("f1_score", best_f1)

    # Log comparison artifacts.
    mlflow.log_artifact("outputs/model_comparison_results.csv")
    mlflow.log_artifact("outputs/model_comparison_results.json")
    mlflow.log_artifact("outputs/model_comparison_f1_chart.png")
    mlflow.log_artifact("outputs/model_comparison_metadata.json")
    mlflow.log_artifact("outputs/best_model_classification_report.txt")

    # Log final selected model with signature and input example.
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from this same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register only the final best model using the same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print model and registry details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)


## 12. Verify Final Artifacts

This section checks whether the important comparison artifacts exist.


In [ ]:
# Define expected artifacts.
expected_artifacts = [
    "outputs/model_comparison_results.csv",
    "outputs/model_comparison_results.json",
    "outputs/model_comparison_f1_chart.png",
    "outputs/model_comparison_metadata.json",
    "outputs/best_model_classification_report.txt",
]

# Verify artifacts.
for artifact_path in expected_artifacts:
    print(artifact_path, "->", os.path.exists(artifact_path))




- Multi-model comparison means training several algorithms under the same conditions.
- Each candidate model should get its own MLflow run.
- This allows easy comparison in MLflow UI.
- The best model should be selected using a clear metric.
- In this notebook, F1-score is the selection metric.
- Only the final best model is logged with signature and registered.
- The comparison table and chart explain why the model was selected.
- This is a common production workflow before model promotion.
